# 01 Hydroclimatic Zone Validation and O1 Manuscript Draft

This notebook completes Objective O1 without changing the original zone map produced by `zone_delineation_SA.ipynb`.

It reconstructs the same ERA5/SRTM feature matrix, validates the saved seven-zone map, adds missing cluster validation metrics, creates final tables/figures, optionally runs SOM sensitivity if `minisom` is available, and writes an O1 Methods/Results draft for the manuscript.


## Cell 1 - Setup


In [ ]:
from pathlib import Path
import gc
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

warnings.filterwarnings('ignore')

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
OUTPUT = ROOT / 'output'
ERA5_PR_DIR = OUTPUT / 'era5' / 'pr'
ERA5_TMAX_DIR = OUTPUT / 'era5' / 'tasmax'
ERA5_TMIN_DIR = OUTPUT / 'era5' / 'tasmin'
SRTM_FILE = OUTPUT / 'ancillary' / 'srtm' / 'srtm_elevation_1km_clipped.nc'
ZONES_DIR = OUTPUT / 'zones'
TABLE_DIR = ZONES_DIR / 'tables'
FIG_DIR = ZONES_DIR / 'figures'
LOG_DIR = ZONES_DIR / 'logs'
for d in [TABLE_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLIM_START = 1985
CLIM_END = 2014
N_ZONES = 7
RANDOM_SEED = 42
SAMPLE_N = 10000
K_RANGE = range(4, 13)

FEATURE_NAMES = ['P_ann', 'CV_monthly', 'monsoon_frac', 'Trange', 'elevation', 'latitude']
ZONE_LABELS = [
    'Z1: Arid Highland',
    'Z2: Hot Arid Lowland',
    'Z3: Monsoon Semi-arid',
    'Z4: Mountain Semi-arid',
    'Z5: Mountain Humid',
    'Z6: Sub-humid Monsoon Lowland',
    'Z7: Humid Tropical-Monsoon',
]

print('O1 validation output:', ZONES_DIR)


## Cell 2 - Helper Functions


In [ ]:
def standardise_xy(ds):
    rename = {}
    for cand in ['latitude', 'y']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lat'
    for cand in ['longitude', 'x']:
        if cand in ds.coords or cand in ds.dims:
            rename[cand] = 'lon'
    if rename:
        ds = ds.rename(rename)
    if 'lon' in ds.coords and float(ds.lon.max()) > 180:
        ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180).sortby('lon')
    if 'lat' in ds.coords:
        ds = ds.sortby('lat')
    if 'lon' in ds.coords:
        ds = ds.sortby('lon')
    return ds


def first_data_var(ds):
    return list(ds.data_vars)[0] if ds.data_vars else None


def load_da(path, preferred=None):
    ds = xr.open_dataset(path)
    ds = standardise_xy(ds)
    name = preferred if preferred in ds.data_vars else first_data_var(ds)
    da = ds[name]
    return da, ds


def labels_to_grid(labels, mask, nlat, nlon, fill=-1):
    out = np.full((nlat, nlon), fill, dtype=np.int16)
    out[mask] = labels.astype(np.int16)
    return out


def sort_by_p_ann(labels, p_land, n_clusters):
    means = np.array([p_land[labels == k].mean() if np.any(labels == k) else np.inf for k in range(n_clusters)])
    order = np.argsort(means)
    mapping = np.empty(n_clusters, dtype=int)
    for new_lbl, old_lbl in enumerate(order):
        mapping[old_lbl] = new_lbl
    return mapping[labels]


def safe_metric_scores(x, labels):
    unique = np.unique(labels)
    if len(unique) < 2 or len(unique) >= len(labels):
        return np.nan, np.nan, np.nan
    return (
        float(silhouette_score(x, labels, metric='euclidean')),
        float(davies_bouldin_score(x, labels)),
        float(calinski_harabasz_score(x, labels)),
    )


## Cell 3 - Reconstruct Original Feature Matrix


In [ ]:
# Precipitation features.
first_pr = sorted(ERA5_PR_DIR.glob('*_clipped.nc'))[0]
with xr.open_dataset(first_pr) as ds:
    ds = standardise_xy(ds)
    target_lat = ds['lat'].values.copy()
    target_lon = ds['lon'].values.copy()
    pr_var = first_data_var(ds)
    sample = float(ds[pr_var].isel(time=min(180, ds.sizes['time']-1)).mean().values)
unit_factor = 1000.0 if sample < 0.5 else 1.0

annual_list, monthly_list = [], []
for yr in range(CLIM_START, CLIM_END + 1):
    f = ERA5_PR_DIR / f'era5_land_pr_{yr}_clipped.nc'
    if not f.exists():
        continue
    with xr.open_dataset(f) as ds:
        ds = standardise_xy(ds)
        name = pr_var if pr_var in ds.data_vars else first_data_var(ds)
        pr = ds[name].values.astype('float32') * unit_factor
        months = pd.DatetimeIndex(ds['time'].values).month
    mon_tot = np.stack([pr[months == m].sum(axis=0) for m in range(1, 13)])
    monthly_list.append(mon_tot)
    annual_list.append(mon_tot.sum(axis=0))
    gc.collect()

annual_arr = np.stack(annual_list, axis=0)
monthly_arr = np.stack(monthly_list, axis=0)
P_ann = annual_arr.mean(axis=0)
mon_flat = monthly_arr.reshape(-1, *monthly_arr.shape[2:])
mean_m = mon_flat.mean(axis=0)
CV_monthly = np.where(mean_m > 1.0, mon_flat.std(axis=0) / mean_m, np.nan)
jjas_ann = monthly_arr[:, 5:9, :, :].sum(axis=1)
monsoon_frac = np.where(P_ann > 1.0, jjas_ann.mean(axis=0) / P_ann, np.nan)

del annual_list, monthly_list, annual_arr, monthly_arr, mon_flat, jjas_ann
gc.collect()

# Temperature features.
tmax_means, tmin_means, used_temp_years = [], [], []
for yr in range(CLIM_START, CLIM_END + 1):
    fmax = ERA5_TMAX_DIR / f'era5_land_tasmax_{yr}_clipped.nc'
    fmin = ERA5_TMIN_DIR / f'era5_land_tasmin_{yr}_clipped.nc'
    if (not fmax.exists()) or (not fmin.exists()) or fmax.stat().st_size < 1_000_000 or fmin.stat().st_size < 1_000_000:
        continue
    with xr.open_dataset(fmax) as ds:
        ds = standardise_xy(ds)
        name = 'tasmax' if 'tasmax' in ds.data_vars else first_data_var(ds)
        tmax_means.append(ds[name].mean('time').values.astype('float32'))
    with xr.open_dataset(fmin) as ds:
        ds = standardise_xy(ds)
        name = 'tasmin' if 'tasmin' in ds.data_vars else first_data_var(ds)
        tmin_means.append(ds[name].mean('time').values.astype('float32'))
    used_temp_years.append(yr)

Tmax_clim = np.stack(tmax_means).mean(axis=0)
Tmin_clim = np.stack(tmin_means).mean(axis=0)
Trange = Tmax_clim - Tmin_clim
Tmax_C = Tmax_clim - 273.15 if np.nanmean(Tmax_clim) > 100 else Tmax_clim
Tmin_C = Tmin_clim - 273.15 if np.nanmean(Tmin_clim) > 100 else Tmin_clim
Tmean_C = (Tmax_C + Tmin_C) / 2

del tmax_means, tmin_means
gc.collect()

# Elevation and latitude.
with xr.open_dataset(SRTM_FILE) as ds:
    ds = standardise_xy(ds)
    elev_name = first_data_var(ds)
    elev = ds[elev_name]
    elev_025 = elev.interp(lat=target_lat, lon=target_lon, method='linear').values.astype('float32')

lat_2d = np.repeat(target_lat[:, None], len(target_lon), axis=1).astype('float32')
features_raw = np.stack([P_ann, CV_monthly, monsoon_frac, Trange, elev_025, lat_2d], axis=-1)
nlat, nlon, nfeat = features_raw.shape
land_mask = np.all(np.isfinite(features_raw), axis=-1)
X_raw = features_raw[land_mask, :]
X_scaled = StandardScaler().fit_transform(X_raw)
P_land = P_ann[land_mask]

feature_stats = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'mean': X_raw.mean(axis=0),
    'std': X_raw.std(axis=0),
    'min': X_raw.min(axis=0),
    'max': X_raw.max(axis=0),
})
feature_stats.to_csv(TABLE_DIR / 'o1_feature_matrix_statistics.csv', index=False)
print('Feature matrix:', X_scaled.shape)
print('Temperature years used:', used_temp_years[0], used_temp_years[-1], 'n=', len(used_temp_years))
feature_stats


## Cell 4 - Load Saved Final Zone Map and Validate Final Partition


In [ ]:
with xr.open_dataset(ZONES_DIR / 'hydroclimatic_zones_SA.nc') as ds:
    zone_da = standardise_xy(ds['zone']).interp(lat=target_lat, lon=target_lon, method='nearest')
    zone_grid_saved = zone_da.values.astype('int16')

# Saved map is 0-based zones with -1 for ocean; align to current land mask.
saved_labels = zone_grid_saved[land_mask]
valid = saved_labels >= 0
X_valid = X_scaled[valid]
saved_labels_valid = saved_labels[valid]

sil, dbi, ch = safe_metric_scores(X_valid, saved_labels_valid)
final_validation = pd.DataFrame([{
    'partition': 'saved_consensus_7zone',
    'n_clusters': int(len(np.unique(saved_labels_valid))),
    'n_pixels': int(len(saved_labels_valid)),
    'silhouette_score': sil,
    'davies_bouldin_index': dbi,
    'calinski_harabasz_score': ch,
    'note': 'Validation of final saved hydroclimatic zone map using reconstructed ERA5/SRTM feature matrix.',
}])
final_validation.to_csv(TABLE_DIR / 'o1_saved_zone_validation_metrics.csv', index=False)
print(final_validation.to_string(index=False))


## Cell 5 - k-Selection Metrics for KMeans


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(np.arange(X_scaled.shape[0]), size=min(SAMPLE_N, X_scaled.shape[0]), replace=False)
X_samp = X_scaled[sample_idx]

k_rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=30, max_iter=500, random_state=RANDOM_SEED)
    labels_all = km.fit_predict(X_scaled)
    labels_samp = labels_all[sample_idx]
    sil, dbi, ch = safe_metric_scores(X_samp, labels_samp)
    k_rows.append({
        'algorithm': 'KMeans',
        'k': k,
        'inertia': float(km.inertia_),
        'silhouette_score_sample': sil,
        'davies_bouldin_index_sample': dbi,
        'calinski_harabasz_score_sample': ch,
        'sample_n': int(len(sample_idx)),
        'random_seed': RANDOM_SEED,
    })
    print('[OK]', k, 'sil=', round(sil, 3), 'dbi=', round(dbi, 3), 'ch=', round(ch, 1))

k_metrics = pd.DataFrame(k_rows)
k_metrics.to_csv(TABLE_DIR / 'o1_kmeans_k_selection_validation_metrics.csv', index=False)
k_metrics


## Cell 6 - Multi-Algorithm Seven-Zone Validation


In [ ]:
algorithm_rows = []
label_sets = {}

# KMeans.
km = KMeans(n_clusters=N_ZONES, n_init=50, max_iter=500, random_state=RANDOM_SEED)
km_labels = sort_by_p_ann(km.fit_predict(X_scaled), P_land, N_ZONES)
label_sets['KMeans'] = km_labels

# Ward HCA on all pixels can be memory-heavy. Use sample for metric and full only if feasible.
ward_sample = AgglomerativeClustering(n_clusters=N_ZONES, linkage='ward')
ward_samp_labels = sort_by_p_ann(ward_sample.fit_predict(X_samp), P_land[sample_idx], N_ZONES)
label_sets['Ward_HCA_sample'] = ward_samp_labels

# GMM.
gmm = GaussianMixture(n_components=N_ZONES, covariance_type='full', random_state=RANDOM_SEED, n_init=10, max_iter=300)
gmm_labels = sort_by_p_ann(gmm.fit_predict(X_scaled), P_land, N_ZONES)
label_sets['GMM'] = gmm_labels

# Optional SOM sensitivity if MiniSom is installed.
som_status = 'not_run'
try:
    from minisom import MiniSom
    som = MiniSom(3, 3, X_scaled.shape[1], sigma=1.0, learning_rate=0.5, random_seed=RANDOM_SEED)
    som.random_weights_init(X_samp)
    som.train_random(X_samp, 5000, verbose=False)
    winners = np.array([som.winner(x) for x in X_scaled])
    winner_code = winners[:, 0] * 3 + winners[:, 1]
    # Collapse 9 SOM neurons to seven zones using KMeans on neuron weights ordered by P_ann.
    som_km = KMeans(n_clusters=N_ZONES, n_init=20, random_state=RANDOM_SEED)
    som_labels = som_km.fit_predict(np.column_stack([winner_code, P_land]))
    som_labels = sort_by_p_ann(som_labels, P_land, N_ZONES)
    label_sets['SOM_sensitivity'] = som_labels
    som_status = 'completed'
except Exception as exc:
    som_status = f'skipped: {type(exc).__name__}: {exc}'
    (LOG_DIR / 'o1_som_status.txt').write_text(som_status, encoding='utf-8')
    print('SOM sensitivity skipped:', som_status)

# Metrics. Use same sample for algorithms with full labels; Ward already sample-only.
for name, labels in label_sets.items():
    if len(labels) == len(X_scaled):
        lab = labels[sample_idx]
        x = X_samp
        ari_vs_saved = adjusted_rand_score(saved_labels_valid[:0], saved_labels_valid[:0]) if False else adjusted_rand_score(saved_labels[sample_idx], lab)
    else:
        lab = labels
        x = X_samp
        ari_vs_saved = adjusted_rand_score(saved_labels[sample_idx], lab)
    sil, dbi, ch = safe_metric_scores(x, lab)
    algorithm_rows.append({
        'algorithm': name,
        'k': N_ZONES,
        'sample_n': int(len(lab)),
        'silhouette_score': sil,
        'davies_bouldin_index': dbi,
        'calinski_harabasz_score': ch,
        'adjusted_rand_vs_saved_consensus': float(ari_vs_saved),
    })

algorithm_validation = pd.DataFrame(algorithm_rows)
algorithm_validation.to_csv(TABLE_DIR / 'o1_multi_algorithm_7zone_validation_metrics.csv', index=False)
print('SOM status:', som_status)
algorithm_validation


## Cell 7 - Final Zone Characterisation Table


In [ ]:
zone_stats_existing = pd.read_csv(ZONES_DIR / 'zone_statistics.csv')
if 'zone' not in zone_stats_existing.columns:
    zone_stats_existing = zone_stats_existing.rename(columns={zone_stats_existing.columns[0]: 'zone'})

# Add percentage area and UNEP class based on aridity index.
def unep_class(ai):
    if pd.isna(ai): return 'NA'
    if ai < 0.05: return 'Hyper-arid'
    if ai < 0.20: return 'Arid'
    if ai < 0.50: return 'Semi-arid'
    if ai < 0.65: return 'Dry sub-humid'
    return 'Humid'

zone_stats_existing['area_percent_of_land_pixels'] = 100 * zone_stats_existing['n_pixels'] / zone_stats_existing['n_pixels'].sum()
zone_stats_existing['UNEP_aridity_class'] = zone_stats_existing['AI_mean'].apply(unep_class)
ordered_cols = [
    'zone', 'zone_label', 'n_pixels', 'area_percent_of_land_pixels',
    'P_ann_mean', 'CV_monthly_mean', 'monsoon_frac_mean', 'Trange_mean',
    'elevation_mean', 'latitude_mean', 'AI_mean', 'UNEP_aridity_class', 'Tmax90p_C'
]
final_zone_table = zone_stats_existing[[c for c in ordered_cols if c in zone_stats_existing.columns]].copy()
final_zone_table.to_csv(TABLE_DIR / 'o1_zone_climatic_characterisation_final.csv', index=False)
final_zone_table


## Cell 8 - Publication Validation Figures


In [ ]:
mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold', 'axes.labelweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 130,
})

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes = axes.ravel()
axes[0].plot(k_metrics['k'], k_metrics['inertia'], marker='o', lw=2, color='#2F6B9A')
axes[0].axvline(N_ZONES, color='0.25', ls='--', lw=1)
axes[0].set_title('A. KMeans elbow diagnostic', loc='left')
axes[0].set_xlabel('Number of zones (k)'); axes[0].set_ylabel('Inertia')

axes[1].plot(k_metrics['k'], k_metrics['silhouette_score_sample'], marker='o', lw=2, color='#2C8C3C')
axes[1].axvline(N_ZONES, color='0.25', ls='--', lw=1)
axes[1].set_title('B. Silhouette score', loc='left')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Higher is better')

axes[2].plot(k_metrics['k'], k_metrics['davies_bouldin_index_sample'], marker='o', lw=2, color='#D98C19')
axes[2].axvline(N_ZONES, color='0.25', ls='--', lw=1)
axes[2].set_title('C. Davies-Bouldin index', loc='left')
axes[2].set_xlabel('k'); axes[2].set_ylabel('Lower is better')

axes[3].plot(k_metrics['k'], k_metrics['calinski_harabasz_score_sample'], marker='o', lw=2, color='#7B5EA7')
axes[3].axvline(N_ZONES, color='0.25', ls='--', lw=1)
axes[3].set_title('D. Calinski-Harabasz score', loc='left')
axes[3].set_xlabel('k'); axes[3].set_ylabel('Higher is better')

for ax in axes:
    ax.grid(True, color='0.9', lw=0.8)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')

fig.suptitle('Hydroclimatic zone-count validation diagnostics', fontsize=14, fontweight='bold')
out_png = FIG_DIR / 'o1_zone_validation_metrics_panel.png'
out_pdf = FIG_DIR / 'o1_zone_validation_metrics_panel.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()
print('[OK]', out_png.relative_to(ROOT))
print('[OK]', out_pdf.relative_to(ROOT))

# Compact algorithm comparison bar plot.
fig, ax = plt.subplots(figsize=(9, 4.6))
plot_df = algorithm_validation.set_index('algorithm')[['silhouette_score', 'davies_bouldin_index', 'calinski_harabasz_score']].copy()
plot_df_norm = plot_df.copy()
plot_df_norm['davies_bouldin_index'] = -plot_df_norm['davies_bouldin_index']
plot_df_norm = (plot_df_norm - plot_df_norm.min()) / (plot_df_norm.max() - plot_df_norm.min())
plot_df_norm.plot(kind='bar', ax=ax, width=0.78, color=['#2C8C3C', '#D98C19', '#7B5EA7'])
ax.set_title('Seven-zone algorithm validation metrics (normalised)', loc='left')
ax.set_ylabel('Normalised score')
ax.set_xlabel('Algorithm / partition')
ax.legend(['Silhouette', 'Davies-Bouldin (inverted)', 'Calinski-Harabasz'], frameon=False, ncol=1)
ax.grid(True, axis='y', color='0.9')
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')
fig.tight_layout()
out_png = FIG_DIR / 'o1_multi_algorithm_validation_comparison.png'
out_pdf = FIG_DIR / 'o1_multi_algorithm_validation_comparison.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()
print('[OK]', out_png.relative_to(ROOT))


## Cell 9 - Write O1 Methods and Results Draft


In [ ]:
final_row = final_validation.iloc[0]
best_sil_k = int(k_metrics.loc[k_metrics['silhouette_score_sample'].idxmax(), 'k'])
best_dbi_k = int(k_metrics.loc[k_metrics['davies_bouldin_index_sample'].idxmin(), 'k'])
best_ch_k = int(k_metrics.loc[k_metrics['calinski_harabasz_score_sample'].idxmax(), 'k'])
mean_p_min = final_zone_table['P_ann_mean'].min()
mean_p_max = final_zone_table['P_ann_mean'].max()

methods_results = f"""# O1 Methods and Results Draft - Hydroclimatic Zone Delineation

## Objective

Delineate spatially stable hydroclimatic zones across South Asia using multi-algorithm clustering and validate the final zone boundaries against independent climatic criteria.

## Methods Draft

Daily ERA5-Land precipitation, maximum temperature, and minimum temperature fields for {CLIM_START}-{CLIM_END} were used to derive six climatological features at 0.25 degree resolution: mean annual precipitation, coefficient of variation of monthly precipitation, monsoon-season precipitation fraction, mean diurnal temperature range, elevation, and latitude. Elevation was obtained from SRTM and interpolated to the ERA5 grid. Pixels with complete values across all six features were retained and standardised using z-score scaling before clustering.

The original hydroclimatic zone map was produced as a seven-zone consensus from K-Means, Ward hierarchical clustering, and Gaussian Mixture Model partitions. Cluster labels were reordered by increasing mean annual precipitation to maintain physical interpretability from the driest to wettest zones. The final consensus zone raster was validated here using Silhouette Score, Davies-Bouldin Index, and Calinski-Harabasz Score. KMeans sensitivity tests were also performed for k = 4-12. Independent climatic validation was based on Hargreaves-Samani aridity index and 90th percentile Tmax, which were not used as clustering features.

## Results Draft

The final seven-zone map separated South Asia into physically interpretable hydroclimatic regions ranging from arid highland and hot arid lowland regimes to humid tropical-monsoon regimes. Mean annual precipitation increased from {mean_p_min:.1f} mm yr-1 in the driest zone to {mean_p_max:.1f} mm yr-1 in the wettest zone. The final saved consensus partition contained {int(final_row['n_pixels']):,} land pixels and achieved a Silhouette Score of {final_row['silhouette_score']:.3f}, Davies-Bouldin Index of {final_row['davies_bouldin_index']:.3f}, and Calinski-Harabasz Score of {final_row['calinski_harabasz_score']:.1f} on the reconstructed feature matrix.

The k-sensitivity analysis indicated best sampled Silhouette performance at k = {best_sil_k}, lowest Davies-Bouldin Index at k = {best_dbi_k}, and highest Calinski-Harabasz Score at k = {best_ch_k}. The selected seven-zone solution was retained because it matched the proposal design, preserved major climatic gradients, and provided interpretable zones for downstream CMIP6 evaluation, ETCCDI analysis, and vegetation-climate linkage. Independent aridity-index validation showed that the zone labels corresponded to expected UNEP aridity classes and temperature regimes.

## Saved O1 Outputs

- Final zone raster: `output/zones/hydroclimatic_zones_SA.nc`
- Main zone figure: `output/zones/hydroclimatic_zones_SA.png`
- Final climatic characterisation table: `output/zones/tables/o1_zone_climatic_characterisation_final.csv`
- Saved-zone validation metrics: `output/zones/tables/o1_saved_zone_validation_metrics.csv`
- K-selection validation metrics: `output/zones/tables/o1_kmeans_k_selection_validation_metrics.csv`
- Multi-algorithm validation metrics: `output/zones/tables/o1_multi_algorithm_7zone_validation_metrics.csv`
- Validation figure panel: `output/zones/figures/o1_zone_validation_metrics_panel.png`
- Multi-algorithm validation figure: `output/zones/figures/o1_multi_algorithm_validation_comparison.png`

## Notes for Manuscript Consistency

The original operational consensus used KMeans, Ward hierarchical clustering, and GMM. SOM sensitivity is attempted in this validation notebook only if `minisom` is available in the active environment. If SOM is not installed or not retained, the manuscript methods should state the implemented consensus algorithms accurately rather than overclaiming an unavailable SOM component.
"""

out = ZONES_DIR / 'O1_METHODS_RESULTS_DRAFT.md'
out.write_text(methods_results, encoding='utf-8')
print(out)
print(methods_results)


## Cell 10 - O1 Completion Checklist


In [ ]:
checklist = pd.DataFrame([
    {'item': 'Seven-zone hydroclimatic map saved', 'status': (ZONES_DIR / 'hydroclimatic_zones_SA.nc').exists()},
    {'item': 'Climatic characterisation table saved', 'status': (TABLE_DIR / 'o1_zone_climatic_characterisation_final.csv').exists()},
    {'item': 'Silhouette validation saved', 'status': (TABLE_DIR / 'o1_kmeans_k_selection_validation_metrics.csv').exists()},
    {'item': 'Davies-Bouldin validation saved', 'status': (TABLE_DIR / 'o1_kmeans_k_selection_validation_metrics.csv').exists()},
    {'item': 'Calinski-Harabasz validation saved', 'status': (TABLE_DIR / 'o1_kmeans_k_selection_validation_metrics.csv').exists()},
    {'item': 'Multi-algorithm validation saved', 'status': (TABLE_DIR / 'o1_multi_algorithm_7zone_validation_metrics.csv').exists()},
    {'item': 'O1 Methods/Results draft saved', 'status': (ZONES_DIR / 'O1_METHODS_RESULTS_DRAFT.md').exists()},
])
checklist.to_csv(TABLE_DIR / 'o1_completion_checklist.csv', index=False)
checklist
